# 00 — Environment Check

Quick smoke test for every external connection this module needs, before
running anything that actually creates data. Run this first any time you
set up a new `.env` or point at a different OneBill/Dataverse/MySQL
environment.

Checks:
1. `.env` has all required keys
2. OneBill OAuth token fetch
3. Dataverse OAuth token fetch
4. MySQL connectivity


In [8]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403


## 1. Required `.env` keys present?

In [9]:
REQUIRED_KEYS = [
    "CRM_TENANT_ID", "CRM_CLIENT_ID", "CRM_CLIENT_SECRET", "CRM_ENVIRONMENT_URL",
    "DB_USERNAME", "DB_PASSWORD", "DB_HOST",
    "CLIENT_ID", "CLIENT_SECRET", "API_USERNAME", "API_PASSWORD",
    "CREATION_PROXY_ACCOUNT_NUMBER",
    "VOYAGER_CCP_KEY", "VOYAGER_PARTNER_ID",
]

missing = [k for k in REQUIRED_KEYS if not os.environ.get(k)]
if missing:
    print(f"MISSING: {missing}")
else:
    print("All required .env keys are present.")

optional = ["MANAGED_BY_WILLIAMS_ACCOUNT_NUMBER", "WILLIAMS_CORPORATION_ACCOUNT_NUMBER", "ONEBILL_BASE_URL"]
for k in optional:
    print(f"  (optional) {k} = {os.environ.get(k) or '<not set — using placeholder/default>'}")


All required .env keys are present.
  (optional) MANAGED_BY_WILLIAMS_ACCOUNT_NUMBER = <not set — using placeholder/default>
  (optional) WILLIAMS_CORPORATION_ACCOUNT_NUMBER = <not set — using placeholder/default>
  (optional) ONEBILL_BASE_URL = https://sandbox-portal.ib.nz


## 2. OneBill OAuth token

In [10]:
try:
    token = token_manager.get_token()
    print(f"OneBill token OK — {token[:12]}... (base url: {ONEBILL_BASE_URL})")
except Exception as e:
    print(f"OneBill token FAILED: {e}")


OneBill token OK — 94370c33-13e... (base url: https://sandbox-portal.ib.nz)


## 3. Dataverse OAuth token

In [11]:
try:
    dv_token = get_dataverse_token()
    print(f"Dataverse token OK — {dv_token[:12]}... (env: {CRM_ENVIRONMENT_URL})")
except Exception as e:
    print(f"Dataverse token FAILED: {e}")


Dataverse token OK — eyJ0eXAiOiJK... (env: https://vygr.crm6.dynamics.com)


## 4. MySQL connectivity

In [12]:
try:
    assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
    from sqlalchemy import create_engine
    engine = create_engine(BI_DATASTORE_URL)
    with engine.connect() as conn:
        conn.exec_driver_sql("SELECT 1")
    print("MySQL connection OK")
except Exception as e:
    print(f"MySQL connection FAILED: {e}")


MySQL connection OK


## 5. Voyager address lookup (circuits + address-search)

Confirms `VOYAGER_CCP_KEY` / `VOYAGER_PARTNER_ID` are set and that the
`NZ_Regions.xlsx` region -> ISO map loaded. This does **not** call the
Voyager API itself (there's no safe/generic SupplierServiceID to test
with here) — the real end-to-end check happens the first time
`05_Fetch_Subscriptions.ipynb` runs `get_voyager_address(...)`.

In [13]:
voyager_keys_ok = bool(VOYAGER_CCP_KEY) and bool(VOYAGER_PARTNER_ID)
print(f"VOYAGER_CCP_KEY set:    {bool(VOYAGER_CCP_KEY)}")
print(f"VOYAGER_PARTNER_ID set: {bool(VOYAGER_PARTNER_ID)}")

if not voyager_keys_ok:
    print("MISSING — set VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID in .env before running 05_Fetch_Subscriptions.ipynb")

if REGION_ISO_MAP:
    print(f"NZ_Regions.xlsx loaded — {len(REGION_ISO_MAP)} region -> ISO mappings")
else:
    print(f"NZ_Regions.xlsx NOT FOUND at {NZ_REGIONS_FILE.resolve()} — region/state on addresses will be blank")


VOYAGER_CCP_KEY set:    True
VOYAGER_PARTNER_ID set: True
NZ_Regions.xlsx loaded — 17 region -> ISO mappings


## 6. Target account numbers

Just a reminder of what's currently configured — these are placeholders
until `04_Create_Accounts.ipynb` has run (or you set the real values in
`.env` / `onebill_common.py`).

In [14]:
TARGET_ACCOUNTS

{'managed_by_williams': {'account_number': 'MANAGED-BY-WILLIAMS',
  'account_name': 'Managed by Williams'},
 'williams_corporation': {'account_number': 'WILLIAMS-CORPORATION',
  'account_name': 'Williams Corporation'}}